In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Memantau atau membatalkan job

Lihat daftar beban kerja kamu di [halaman Workloads](https://quantum.cloud.ibm.com/workloads).

## Lihat status job
Buka [tabel Workloads](https://quantum.cloud.ibm.com/workloads) dan periksa kolom Status untuk melihat apakah job sudah selesai atau gagal.

## Lihat sisa penggunaan
Buka [tabel Instances](https://quantum.cloud.ibm.com/instances) dan pilih tab yang sesuai dengan paket yang ingin kamu lihat sisa penggunaannya. Total waktu yang sudah digunakan dan total waktu yang tersisa pada paket kamu akan ditampilkan.

## Lihat metrik jumlah job dan workload yang dikirimkan
Buka [halaman Analytics](https://quantum.cloud.ibm.com/analytics) untuk melihat total jumlah job yang dikirimkan, serta jumlah batch workload dan session workload. Perlu diingat bahwa kamu hanya bisa melihat halaman Analytics untuk akun yang kamu miliki atau kelola.

## Pantau job
Gunakan instance job untuk memeriksa status job atau mengambil hasilnya dengan memanggil perintah yang sesuai:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | Tinjau hasil job segera setelah job selesai. Hasil job tersedia setelah job selesai. Oleh karena itu, job.result() adalah panggilan pemblokiran sampai job selesai.                               |
| job.job\_id()                 | Kembalikan ID yang secara unik mengidentifikasi job tersebut. Mengambil hasil job di lain waktu membutuhkan ID job. Oleh karena itu, disarankan untuk menyimpan ID job yang mungkin ingin kamu ambil nanti. |
| job.status()                  | Periksa status job.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | Ambil job yang sebelumnya kamu kirimkan. Panggilan ini membutuhkan ID job.                                                                                                                                      |

<span id="retrieve-later"></span>
## Ambil hasil job di lain waktu
Panggil `service.job(\<job\_id>)` untuk mengambil job yang sebelumnya kamu kirimkan. Jika kamu tidak punya ID job, atau jika ingin mengambil beberapa job sekaligus — termasuk job dari QPU (unit pemrosesan kuantum) yang sudah pensiun — panggil `service.jobs()` dengan filter opsional. Lihat [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` juga mengembalikan job yang dijalankan dari paket `qiskit-ibm-provider` yang sudah tidak didukung. Job yang dikirimkan oleh paket yang lebih lama (juga sudah tidak didukung) `qiskit-ibmq-provider` tidak lagi tersedia.

### Contoh
Contoh ini mengembalikan 10 runtime job terbaru yang dijalankan di `my_backend`:

In [1]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

<span id="execution-spans"></span>
## View Sampler execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

Example:

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`:

In [6]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [7]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])

View global information about the collection of execution spans:

In [8]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2025-09-09 16:31:16.320568
     End of the last span: 2025-09-09 16:31:16.865858
       Total duration (s): 0.54529


Extract and inspect a particular span:

In [9]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2025-09-09 16:31:16.320568
   End of first span: 2025-09-09 16:31:16.865858
#shots in first span: 24


## Batalkan job
Kamu bisa membatalkan job dari dasbor IBM Quantum Platform baik di halaman Workloads maupun halaman detail untuk workload tertentu. Di halaman Workloads, klik menu overflow di akhir baris workload tersebut, lalu pilih Cancel. Jika kamu berada di halaman detail untuk workload tertentu, gunakan dropdown Actions di bagian atas halaman, lalu pilih Cancel.

Di Qiskit, gunakan `job.cancel()` untuk membatalkan job.

<span id="execution-spans"></span>
## Lihat execution spans Sampler
Hasil job [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) yang dieksekusi di Qiskit Runtime mengandung informasi waktu eksekusi dalam metadata-nya.
Informasi waktu ini bisa digunakan untuk menetapkan batas timestamp atas dan bawah kapan shot tertentu dieksekusi di QPU.
Shot dikelompokkan ke dalam objek [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span), masing-masing menunjukkan waktu mulai, waktu berhenti, dan spesifikasi shot mana yang dikumpulkan dalam span tersebut.

Sebuah execution span menentukan data mana yang dieksekusi selama jendela waktunya dengan menyediakan metode [`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask). Metode ini, dengan indeks [Primitive Unified Block (PUB)](/guides/primitive-input-output) mana pun, mengembalikan boolean mask yang bernilai `True` untuk semua shot yang dieksekusi selama jendela waktunya. PUB diindeks berdasarkan urutan pemberiannya pada pemanggilan Sampler run. Jika, misalnya, sebuah PUB berbentuk `(2, 3)` dan dijalankan dengan empat shot, maka bentuk mask-nya adalah `(2, 3, 4)`. Lihat halaman API [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span) untuk detail lengkap.

Contoh:
Untuk melihat informasi execution span, tinjau metadata hasil yang dikembalikan oleh `SamplerV2`, yang hadir dalam bentuk objek `ExecutionSpans`. Objek ini adalah kontainer mirip-list yang berisi instance dari subkelas `ExecutionSpan`, seperti `SliceSpan`: